In [1]:
"""
Correlation-Augmented Fusion Dataset Preparation
=================================================
For each image:
  1. Load 4 modality channels (reflec, signal, nearir, range)
  2. Compute 6 local correlation maps between all channel pairs
     using a sliding window (patch-based Pearson correlation)
  3. Stack original 4 channels + 6 correlation maps = 10 channels
  4. Project 10 → 3 via PCA fitted on training data
  5. Save as PNG

Output: dataset_corr_fusion/images/{train,valid,test}/*.png
"""

import cv2
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from scipy.ndimage import uniform_filter
from tqdm import tqdm

# ── CONFIG ──────────────────────────────────────────────────
ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"

OUT_ROOT  = Path("dataset_corr_fusion/images")
MODALITIES = ["reflec", "signal", "nearir", "range"]  # 4 channels
SPLITS     = ["train", "valid", "test"]
IMG_EXTS   = [".png", ".jpg", ".jpeg"]

WINDOW    = 9      # local correlation window size (pixels)
MAX_PX    = 200_000
np.random.seed(42)
# ────────────────────────────────────────────────────────────


def read_channel(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    if img.ndim == 3:
        img = img[:, :, 0]
    return img.astype(np.float32)


def load_stack(img_name: str, split: str) -> np.ndarray:
    """Returns H x W x 4."""
    return np.stack(
        [read_channel(ROOT / mod / split / img_name) for mod in MODALITIES],
        axis=-1
    )


def local_corr(a: np.ndarray, b: np.ndarray, win: int) -> np.ndarray:
    """
    Local Pearson correlation between channels a and b.
    Uses efficient sliding window via uniform_filter.
    Returns H x W map in [-1, 1].
    """
    w = float(win * win)

    mu_a  = uniform_filter(a,   size=win)
    mu_b  = uniform_filter(b,   size=win)
    mu_aa = uniform_filter(a*a, size=win)
    mu_bb = uniform_filter(b*b, size=win)
    mu_ab = uniform_filter(a*b, size=win)

    cov  = mu_ab - mu_a * mu_b
    std_a = np.sqrt(np.maximum(mu_aa - mu_a**2, 1e-8))
    std_b = np.sqrt(np.maximum(mu_bb - mu_b**2, 1e-8))

    corr = cov / (std_a * std_b + 1e-8)
    return np.clip(corr, -1.0, 1.0)


def build_10ch(stack: np.ndarray) -> np.ndarray:
    """
    stack: H x W x 4
    Returns: H x W x 10
      channels 0-3  : original modalities (normalised)
      channels 4-9  : local correlation maps for all 6 pairs
        4: reflec  x signal
        5: reflec  x nearir
        6: reflec  x range
        7: signal  x nearir
        8: signal  x range
        9: nearir  x range
    """
    H, W, _ = stack.shape

    # Normalise each channel to [0,1] for stable correlation
    normed = np.zeros_like(stack)
    for c in range(4):
        ch = stack[:, :, c]
        mn, mx = ch.min(), ch.max()
        normed[:, :, c] = (ch - mn) / (mx - mn + 1e-6)

    pairs = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
    corr_maps = []
    for i, j in pairs:
        corr_maps.append(local_corr(normed[:,:,i], normed[:,:,j], WINDOW))

    corr_stack = np.stack(corr_maps, axis=-1)  # H x W x 6

    return np.concatenate([normed, corr_stack], axis=-1)  # H x W x 10


# ══════════════════════════════════════════════════════════════
# STEP 1 — Sample pixels from training set to fit PCA
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1: Sampling training pixels for PCA fitting...")
print("=" * 60)

train_dir   = ROOT / MODALITIES[0] / "train"
train_imgs  = [p for p in train_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

samples = []
for img_path in tqdm(train_imgs, desc="Sampling"):
    stack  = load_stack(img_path.name, "train")
    ten_ch = build_10ch(stack)                      # H x W x 10
    pixels = ten_ch.reshape(-1, 10)

    n = min(len(pixels), 500)
    samples.append(pixels[np.random.choice(len(pixels), n, replace=False)])

samples = np.concatenate(samples)
if len(samples) > MAX_PX:
    samples = samples[np.random.choice(len(samples), MAX_PX, replace=False)]

print(f"  Total pixels for PCA: {len(samples)}")


# ══════════════════════════════════════════════════════════════
# STEP 2 — Fit PCA (10 → 3)
# ══════════════════════════════════════════════════════════════
print("\nSTEP 2: Fitting PCA (10 → 3)...")

pca = PCA(n_components=3)
pca.fit(samples)
print(f"  Explained variance : {np.round(pca.explained_variance_ratio_, 4)}")
print(f"  Total captured     : {pca.explained_variance_ratio_.sum():.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 3 — Global normalisation stats from train
# ══════════════════════════════════════════════════════════════
print("\nSTEP 3: Computing global normalisation stats...")

global_lo = np.full(3, np.inf)
global_hi = np.full(3, -np.inf)

for img_path in tqdm(train_imgs, desc="Stats"):
    stack  = load_stack(img_path.name, "train")
    ten_ch = build_10ch(stack)
    proj   = pca.transform(ten_ch.reshape(-1, 10)).reshape(
        ten_ch.shape[0], ten_ch.shape[1], 3
    )
    for c in range(3):
        global_lo[c] = min(global_lo[c], np.percentile(proj[:,:,c], 1))
        global_hi[c] = max(global_hi[c], np.percentile(proj[:,:,c], 99))

print(f"  Global lo: {np.round(global_lo, 4)}")
print(f"  Global hi: {np.round(global_hi, 4)}")

np.save("corr_pca_components.npy", pca.components_)
np.save("corr_pca_mean.npy",       pca.mean_)
np.save("corr_global_lo.npy",      global_lo)
np.save("corr_global_hi.npy",      global_hi)
print("  Saved transform files.")


# ══════════════════════════════════════════════════════════════
# STEP 4 — Write fused images
# ══════════════════════════════════════════════════════════════
def fuse(stack: np.ndarray) -> np.ndarray:
    """H x W x 4  →  H x W x 3  uint8."""
    ten_ch = build_10ch(stack)
    proj   = pca.transform(ten_ch.reshape(-1, 10)).reshape(
        ten_ch.shape[0], ten_ch.shape[1], 3
    )
    def norm(ch, lo, hi):
        return ((np.clip(ch, lo, hi) - lo) / (hi - lo + 1e-6) * 255).astype(np.uint8)

    r = norm(proj[:,:,0], global_lo[0], global_hi[0])
    g = norm(proj[:,:,1], global_lo[1], global_hi[1])
    b = norm(proj[:,:,2], global_lo[2], global_hi[2])
    return cv2.merge([b, g, r])  # OpenCV BGR


print("\nSTEP 4: Writing fused images...")

for split in SPLITS:
    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    ref_dir    = ROOT / MODALITIES[0] / split
    split_imgs = [p for p in ref_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

    for img_path in tqdm(split_imgs, desc=split):
        stack = load_stack(img_path.name, split)
        fused = fuse(stack)
        cv2.imwrite(str(out_dir / (img_path.stem + ".png")), fused)

print(f"\n✅ Done. Dataset saved to: {OUT_ROOT}")
print("""
Channel layout of the 10-ch intermediate:
  0 = reflec    (normalised)
  1 = signal    (normalised)
  2 = nearir    (normalised)
  3 = range     (normalised)
  4 = corr(reflec,  signal)
  5 = corr(reflec,  nearir)
  6 = corr(reflec,  range)
  7 = corr(signal,  nearir)
  8 = corr(signal,  range)
  9 = corr(nearir,  range)
""")

STEP 1: Sampling training pixels for PCA fitting...


Sampling: 100%|██████████| 1367/1367 [01:19<00:00, 17.24it/s]


  Total pixels for PCA: 200000

STEP 2: Fitting PCA (10 → 3)...
  Explained variance : [0.3647 0.2215 0.1697]
  Total captured     : 0.7559

STEP 3: Computing global normalisation stats...


Stats: 100%|██████████| 1367/1367 [01:22<00:00, 16.50it/s]


  Global lo: [-0.7995 -1.6207 -1.3389]
  Global hi: [1.1663 1.1221 1.3661]
  Saved transform files.

STEP 4: Writing fused images...


test: 100%|██████████| 197/197 [00:12<00:00, 15.59it/s]


✅ Done. Dataset saved to: dataset_corr_fusion\images

Channel layout of the 10-ch intermediate:
  0 = reflec    (normalised)
  1 = signal    (normalised)
  2 = nearir    (normalised)
  3 = range     (normalised)
  4 = corr(reflec,  signal)
  5 = corr(reflec,  nearir)
  6 = corr(reflec,  range)
  7 = corr(signal,  nearir)
  8 = corr(signal,  range)
  9 = corr(nearir,  range)



In [1]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="correlation_maps.yaml",
    imgsz=1024,
    epochs=500,
    patience=80,        # early stopping
    batch=8,
    device=0,
    project="correlation_maps",
    name="correlation_maps",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=correlation_maps.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025F0C930F10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480